In [1]:
import pandas as pd
import duckdb
import os

conn = duckdb.connect('../database/olist.db')

# Orders

In [21]:
# reading data FROM orders


# Execute and fetch as list of tuples

# Convert the base columns to real timestamps one by one
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE orders ALTER {col} TYPE TIMESTAMP USING TRY_CAST({col} AS TIMESTAMP)")

result = conn.execute("SELECT * FROM orders").df().head()
print(result)  # [(2,)]


Optimizing order_purchase_timestamp...
Optimizing order_approved_at...
Optimizing order_delivered_carrier_date...
Optimizing order_delivered_customer_date...
Optimizing order_estimated_delivery_date...
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06 2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39 2018

In [22]:
# calculating number of days till the approval, shipment, delivery

query = """ 
CREATE OR REPLACE TABLE orders_delivery_details AS 
SELECT 
    order_id, customer_id, order_status,
    order_purchase_timestamp,
    order_approved_at,
    datediff('day', order_purchase_timestamp, order_approved_at) as days_till_approval,
    order_delivered_carrier_date,
    datediff('day', order_purchase_timestamp, order_delivered_carrier_date ) as days_till_shipped,
    order_estimated_delivery_date,
    datediff('day', order_purchase_timestamp , order_estimated_delivery_date ) as days_till_estim_delivery,
    order_delivered_customer_date,
    datediff('day', order_purchase_timestamp , order_delivered_customer_date ) as days_till_delivered,
    datediff('day', order_estimated_delivery_date, order_delivered_customer_date) AS delivery_days_accuracy
FROM orders """
conn.execute(query)


In [23]:
# reading the order delivery details
query = """
    SELECT * FROM orders_delivery_details
"""
df=conn.execute(query).df()
print(df)

                               order_id                       customer_id  \
0      e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1      53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2      47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3      949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4      ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
...                                 ...                               ...   
99436  9c5dedf39a927c1b2549525ed64a053c  39bd1228ee8140590ac3aca26f2dfe00   
99437  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
99438  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
99439  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
99440  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   

      order_status order_purchase_timestamp   order_approved_at  \
0       

# Products

In [24]:
# reading data FROM products

# Execute and fetch as list of tuples
result = conn.execute("SELECT * FROM products").df().head()

# Convert the base columns to integers one by one
cols = [
    'product_name_lenght', 
    'product_description_lenght', 
    'product_photos_qty', 
    'product_weight_g', 
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for col in cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE products ALTER {col} TYPE INTEGER USING TRY_CAST({col} AS INTEGER)")

Optimizing product_name_lenght...
Optimizing product_description_lenght...
Optimizing product_photos_qty...
Optimizing product_weight_g...
Optimizing product_length_cm...
Optimizing product_height_cm...
Optimizing product_width_cm...


# Order Items

In [25]:
# reading data FROM order items

# Convert the base columns to float one by one
cols = [
    'price',
    'freight_value'
]

for col in cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE order_items ALTER {col} TYPE FLOAT USING TRY_CAST({col} AS FLOAT)")

# Convert the base column to date columns
col = 'shipping_limit_date'
print(f"Optimizing {col}...")
conn.execute(f"ALTER TABLE order_items ALTER {col} TYPE TIMESTAMP USING TRY_CAST({col} AS TIMESTAMP)")

# Execute and fetch as list of tuples
result = conn.execute("SELECT * FROM order_items").df().head()
print(result)


Optimizing price...
Optimizing freight_value...
Optimizing shipping_limit_date...
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   

  shipping_limit_date       price  freight_value  
0 2017-09-19 09:45:35   58.900002      13.290000  
1 2017-05-03 11:05:13  239.899994      19.930000  
2

In [26]:
# aggregating the order items data to calculate total cost, freight value, shipping burden and total cost to customer
query = """
    CREATE OR REPLACE TABLE order_cost_details AS 
    SELECT 
        order_id, shipping_limit_date, 
        count(distinct product_id) as total_products, 
        count(order_item_id) as total_items, 
        ROUND(SUM(price),4) as order_price, 
        ROUND(SUM(freight_value),4) as order_freight_value,
        ROUND(SUM(price+freight_value),4) as order_cost,
        ROUND(SUM(freight_value/price+freight_value),4) as shipping_burden
    FROM order_items
    GROUP BY order_id, shipping_limit_date       
"""
conn.execute(query)

query = """
    SELECT * FROM order_cost_detils
"""
res = conn.execute(query).df().head()
print(res)

                           order_id shipping_limit_date  total_products  \
0  000e63d38ae8c00bbcb5a30573b99628 2018-03-29 20:07:49               1   
1  0040a56893444cb56cba7cfe2225e34e 2018-01-29 00:15:10               1   
2  004ca5ae248069d68e8594df8abf6ce0 2018-06-14 18:52:39               1   
3  006cba07f62f921fe4f58365bde2b2eb 2017-08-20 21:05:07               1   
4  00c81b6675ef9ecaaff5fc1d6720b96c 2017-10-30 18:14:26               1   

   total_items  total_cost  total_freight_value  total_cost_customer  \
0            1       47.90                 8.88                56.78   
1            1       84.90                34.39               119.29   
2            1      159.90                61.18               221.08   
3            1       69.99                11.99                81.98   
4            1       24.99                14.10                39.09   

   shipping_burden  
0           0.1854  
1           0.4051  
2           0.3826  
3           0.1713  
4          

In [27]:
# aggregating order items data to specifically calculate seller related information
query = """
    CREATE OR REPLACE TABLE seller_performance AS 
    SELECT
        seller_id, 
        product_id,
        order_id,
        count(distinct order_id) as orders_total,
        count(distinct product_id) as across_products,
        ROUND(SUM(price),4) as price_value,
        ROUND(SUM(freight_value),4) as freight_value,
        ROUND(SUM(price+freight_value),4) as cost_total,
        ROUND(SUM(freight_value/price + freight_value),4) as shipping_burden
    FROM order_items
    GROUP BY seller_id, product_id, order_id

   
"""
conn.execute(query)

query = """
    SELECT * FROM seller_performance
"""
res = conn.execute(query).df().head()
print(res)

                          seller_id                        product_id  \
0  92eb0f42c21942b6552362b9b114707d  23365beed316535b4105bd800c46670e   
1  955fee9216a65b617aa5c0531780ce60  35537536ed2b4c561b4018bf3abf54e0   
2  7c67e1448b00f6e969d365cea6b010ab  4c1bbc12438daec98a77243c2bf7a3ba   
3  f80edd2c5aaa505cc4b0a3b219abf4b8  452f66a0f164cac57802e2cea93188ac   
4  7210cd29727d674c00741e5e387b3ccd  0c4a0f8ab44f9acd2d04e7024f9ba362   

                           order_id  orders_total  across_products  \
0  0014ae671de39511f7575066200733b7             1                1   
1  0025c5d1a8ca53a240ec2634bb4492ea             1                1   
2  0036757472ece3dde52fd4bfd929c90e             1                1   
3  003a94f778ef8cfd50247c8c1b582257             1                1   
4  00526a9d4ebde463baee25f386963ddc             1                1   

   price_value  freight_value  cost_total  shipping_burden  
0        16.50          14.10       30.60          14.9545  
1       390.00    

# Order Payments

In [28]:

result = conn.execute("SELECT * FROM order_payments").df().head()
print(result)


                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card   
4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card   

   payment_installments  payment_value  
0                     8          99.33  
1                     1          24.39  
2                     1          65.71  
3                     8         107.78  
4                     2         128.45  


In [29]:
# aggregating payment information about orders
query = """
    CREATE OR REPLACE TABLE order_payments_agg AS
    SELECT
        order_id, 
            SUM(payment_value) as total_payment_value,
            MAX(payment_installments) as max_pay_install,
            COUNT(DISTINCT payment_type) as type_of_payments,
            SUM(CASE WHEN payment_type = 'credit_card' THEN 1 ELSE 0 END) as credit_card_count,
            SUM(CASE WHEN payment_type = 'debit_card' THEN 1 ELSE 0 END) as debit_card_count,
            SUM(CASE WHEN payment_type = 'voucher' THEN 1 ELSE 0 END) as voucher_count,
            SUM(CASE WHEN payment_type = 'boleto' THEN 1 ELSE 0 END) as boleto_count,
            SUM(CASE WHEN payment_type = 'not_defined' THEN 1 ELSE 0 END) as undef_pay_type
    FROM order_payments
    GROUP BY order_id


"""
conn.execute(query)

# Order Reviews

In [30]:
result = conn.execute("SELECT * FROM order_reviews").df()
print(result)

                              review_id                          order_id  \
0      7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1      80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2      228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3      e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4      f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   
...                                 ...                               ...   
99219  574ed12dd733e5fa530cfd4bbf39d7c9  2a8c23fee101d4d5662fa670396eb8da   
99220  f3897127253a9592a73be9bdfdf4ed7a  22ec9f0669f784db00fa86d035cf8602   
99221  b3de70c89b1510c4cd3d0649fd302472  55d4004744368f5571d1f590031933e4   
99222  1adeb9d84d72fe4e337617733eb85149  7725825d039fc1f0ceb7635e3f7d9206   
99223  efe49f1d6f951dd88b51e6ccd4cc548f  90531360ecb1eec2a1fbb265a0db0508   

       review_score review_comment_title  \
0                 4            

In [31]:
# aggregating reviews data on order basis
query = """
CREATE OR REPLACE TABLE  order_review_summary AS
    SELECT 
        order_id, count(review_id) as total_reviews_count,
        ROUND(avg(review_score),2) as avg_score,
        MAX(review_score) as max_score,
        MIN(review_score) as min_score,
        COUNT(review_comment_message ) as textual_reviews
    FROM order_reviews
    GROUP BY order_id
    ORDER BY total_reviews_count desc, avg_score desc
"""
conn.execute(query)

res = conn.execute("SELECT * FROM order_review_summary").df().head()
print(result)

                              review_id                          order_id  \
0      7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1      80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2      228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3      e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4      f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   
...                                 ...                               ...   
99219  574ed12dd733e5fa530cfd4bbf39d7c9  2a8c23fee101d4d5662fa670396eb8da   
99220  f3897127253a9592a73be9bdfdf4ed7a  22ec9f0669f784db00fa86d035cf8602   
99221  b3de70c89b1510c4cd3d0649fd302472  55d4004744368f5571d1f590031933e4   
99222  1adeb9d84d72fe4e337617733eb85149  7725825d039fc1f0ceb7635e3f7d9206   
99223  efe49f1d6f951dd88b51e6ccd4cc548f  90531360ecb1eec2a1fbb265a0db0508   

       review_score review_comment_title  \
0                 4            

In [32]:
# check for orphan ids
query = """ 
    select * from seller_performance

"""
conn.execute(query).df()

,seller_id,product_id,order_id,orders_total,across_products,price_value,freight_value,cost_total,shipping_burden
0,92eb0f42c21942b6552362b9b114707d,23365beed316535b4105bd800c46670e,0014ae671de39511f7575066200733b7,1,1,16.50,14.10,30.60,14.9545
1,955fee9216a65b617aa5c0531780ce60,35537536ed2b4c561b4018bf3abf54e0,0025c5d1a8ca53a240ec2634bb4492ea,1,1,390.00,29.39,419.39,29.4654
2,7c67e1448b00f6e969d365cea6b010ab,4c1bbc12438daec98a77243c2bf7a3ba,0036757472ece3dde52fd4bfd929c90e,1,1,136.99,66.04,203.03,66.5221
3,f80edd2c5aaa505cc4b0a3b219abf4b8,452f66a0f164cac57802e2cea93188ac,003a94f778ef8cfd50247c8c1b582257,1,1,39.90,18.08,57.98,18.5331
4,7210cd29727d674c00741e5e387b3ccd,0c4a0f8ab44f9acd2d04e7024f9ba362,00526a9d4ebde463baee25f386963ddc,1,1,135.56,33.60,169.16,34.5914
...,...,...,...,...,...,...,...,...,...
102420,a08692680c77d30a0b4280da5df01c5a,e86b81dcac341ea01df0260077cdf082,ffb18bf111fa70edf316eb0390427986,1,1,99.00,9.65,108.65,9.7475
102421,dbb9b48c841a0e39e21f98e1a6b2ec3e,f893b82f0fc3ba3686ca2d65eaed5a00,ffbae9b91be0cb7c3cc9a38a35c29210,1,1,44.99,15.10,60.09,15.4356
102422,897060da8b9a21f655304d50fd935913,c3f6113d5b61bc95468432072b27e23d,ffdc30c0eaeed2e08818c4dad9806292,1,1,18.90,15.10,34.00,15.8989
102423,7ea5bfa6c340f58f8e71fc1f0412b0d6,acc78c9d340dbe682491c9d31ea7e187,ffe8851012fcdaf83de7b595fd5154b3,1,1,149.99,21.85,171.84,21.9957


In [ ]:
# Let's focus on delivered orders for now
query = """
    CREATE OR REPLACE VIEW mart_orders_delivered AS
SELECT
    *
FROM orders_delivery_details
WHERE order_status = 'delivered'

"""
conn.execute(query)


In [6]:
query = """ 
    CREATE OR REPLACE VIEW  seller_aggregated AS
    with seller_agg as (
        select 
            seller_id, count(order_id) as total_orders, sum(cost_total) as total_cost
        from seller_performance
        group by seller_id
    ),
    ranked as (
        select *,
            RANK() OVER(order by total_orders desc) as order_rank,
            sum(total_orders) OVER() as grand_total_orders, 
            sum(total_cost) OVER () as grand_total_cost
        from seller_agg
    )
    select 
        seller_id, total_orders, total_cost,
        ROUND(total_orders*100/grand_total_orders) as pct_orders,
        ROUND(100*SUM(total_orders) OVER (ORDER BY total_orders DESC) 
          / grand_total_orders, 2) AS cumulative_pct
    from ranked
    order by total_orders desc
"""
conn.execute(query)

In [7]:
#### Storing the mart data as an object
marts = ['view_orders_delivered', 
         'order_cost_details','customers',
         'products', 'sellers', 'seller_performance', 
         'order_payments_agg', 'order_review_summary','seller_aggregated']
for m in range(0, len(marts)):
    query = f""" 
        select * from {marts[m]}
    """
    df = conn.execute(query).df()
    df.to_parquet(f'../data/marts/{marts[m]}.parquet', index = False)


In [35]:
conn.close()